# Constructing an Index-Based Deep Lake Vector Store for Semantic Search with LlamaIndex and OpenAI

copyright 2024, Denis Rothman

A Practical Guide to Building a Semantic Search Engine with Deep Lake, LlamaIndex, and OpenAI:

*   Installing the Environment
*   Creating and populating the Vector Store &   dataset
*   Getting started with  index-based semantic search




# Installing the environment

In [13]:
#Google Drive option to store API Keys
#Store you key in a file and read it(you can type it directly in the notebook but it will be visible for somebody next to you)
# from google.colab import drive
# drive.mount('/content/drive')

*First run the following cells and restart Google Colab session if prompted. Then run the notebook again cell by cell to explore the code.*

In [14]:
# !pip install llama-index-vector-stores-deeplake==0.1.6

LlamaIndex supports Deep Lake vector stores through the DeepLakeVectorStore class.

In [15]:
# !pip install deeplake==3.9.18

In [16]:
# !pip install llama-index==0.10.64

Next, let's import the required modules and set the needed environmental variables:

In [17]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
from llama_index.vector_stores.deeplake import DeepLakeVectorStore

In [18]:
# !pip install sentence-transformers==3.0.1

In [19]:
#Retrieving and setting the OpenAI API key
f = open("../openai_api_key.txt", "r")
API_KEY=f.readline().strip()
f.close()

#The OpenAI KeyActiveloop and OpenAI API keys
import os
import openai
os.environ['OPENAI_API_KEY'] =API_KEY
openai.api_key = os.getenv("OPENAI_API_KEY")

In [20]:
#Retrieving and setting the Activeloop API token
f = open("../activeloop_api_key.txt", "r")
API_token=f.readline().strip()
f.close()
ACTIVELOOP_TOKEN=API_token
os.environ['ACTIVELOOP_TOKEN'] =ACTIVELOOP_TOKEN

In [21]:
# For Google Colab and Activeloop while waiting for Activeloop (April 2024) pending new version
# This line writes the string "nameserver 8.8.8.8" to the file. This is specifying that the DNS server the system
# should use is at the IP address 8.8.8.8, which is one of Google's Public DNS servers.
# with open('/etc/resolv.conf', 'w') as file:
#     file.write("nameserver 8.8.8.8")

# Pipeline 1 : Collecting and preparing the documents

In [22]:
!mkdir data

mkdir: cannot create directory ‘data’: File exists


In [23]:
import requests
from bs4 import BeautifulSoup
import re
import os

urls = [
    "https://github.com/VisDrone/VisDrone-Dataset",
    "https://paperswithcode.com/dataset/visdrone",
    "https://openaccess.thecvf.com/content_ECCVW_2018/papers/11133/Zhu_VisDrone-DET2018_The_Vision_Meets_Drone_Object_Detection_in_Image_Challenge_ECCVW_2018_paper.pdf",
    "https://github.com/VisDrone/VisDrone2018-MOT-toolkit",
    "https://en.wikipedia.org/wiki/Object_detection",
    "https://en.wikipedia.org/wiki/Computer_vision",
    "https://en.wikipedia.org/wiki/Convolutional_neural_network",
    "https://en.wikipedia.org/wiki/Unmanned_aerial_vehicle",
    "https://www.faa.gov/uas/",
    "https://www.tensorflow.org/",
    "https://pytorch.org/",
    "https://keras.io/",
    "https://arxiv.org/abs/1804.06985",
    "https://arxiv.org/abs/2202.11983",
    "https://motchallenge.net/",
    "http://www.cvlibs.net/datasets/kitti/",
    "https://www.dronedeploy.com/",
    "https://www.dji.com/",
    "https://arxiv.org/",
    "https://openaccess.thecvf.com/",
    "https://roboflow.com/",
    "https://www.kaggle.com/",
    "https://paperswithcode.com/",
    "https://github.com/"
]

In [24]:
import requests
import re
import os
from bs4 import BeautifulSoup

def clean_text(content):
    # Remove references and unwanted characters
    content = re.sub(r'\[\d+\]', '', content)   # Remove references
    content = re.sub(r'[^\w\s\.]', '', content)  # Remove punctuation (except periods)
    return content

def fetch_and_clean(url):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise exception for bad responses (e.g., 404)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Prioritize "mw-parser-output" but fall back to "content" class if not found
        content = soup.find('div', {'class': 'mw-parser-output'}) or soup.find('div', {'id': 'content'})
        if content is None:
            return None

        # Remove specific sections, including nested ones
        for section_title in ['References', 'Bibliography', 'External links', 'See also', 'Notes']:
            section = content.find('span', id=section_title)
            while section:
                for sib in section.parent.find_next_siblings():
                    sib.decompose()
                section.parent.decompose()
                section = content.find('span', id=section_title)

        # Extract and clean text
        text = content.get_text(separator=' ', strip=True)
        text = clean_text(text)
        return text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching content from {url}: {e}")
        return None  # Return None on error

# Directory to store the output files
output_dir = './data/'  # More descriptive name
os.makedirs(output_dir, exist_ok=True)

# Processing each URL (and skipping invalid ones)
for url in urls:
    article_name = url.split('/')[-1].replace('.html', '')  # Handle .html extension
    filename = os.path.join(output_dir, f"{article_name}.txt")

    clean_article_text = fetch_and_clean(url)
    if clean_article_text:  # Only write to file if content exists
        with open(filename, 'w', encoding='utf-8') as file:
            file.write(clean_article_text)

print(f"Content(ones that were possible) written to files in the '{output_dir}' directory.")

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Content(ones that were possible) written to files in the './data/' directory.


In [25]:
# load documents
documents = SimpleDirectoryReader("./data/").load_data()

In [26]:
documents[0]

Document(id_='2e2f6b92-57c2-4663-af82-31c529072c09', embedding=None, metadata={'file_path': '/home/tikim/code/RAG-Driven-Generative-AI/Chapter03/data/1804.06985.txt', 'file_name': '1804.06985.txt', 'file_type': 'text/plain', 'file_size': 3798, 'creation_date': '2025-04-23', 'last_modified_date': '2025-04-23'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text='High Energy Physics  Theory arXiv1804.06985 hepth Submitted on 19 Apr 2018 Title A Near Horizon Extreme Binary Black Hole Geometry Authors Jacob Ciafre  Maria J. Rodriguez View a PDF of the paper titled A Near Horizon Extreme Binary Black Hole Geometry by Jacob Ciafre and Maria J. Rodriguez View PDF Abstract A new solution of fourdimensional vacuum General Relativity is presented. It describes the nea

# Pipeline 2 : Creating and populating a Deep Lake Vector Store

**Replace `hub://denis76/drone_v2` by your organization and dataset name**

In [27]:
from llama_index.core import StorageContext

vector_store_path = "hub://sappho192/drone_v2"
dataset_path = "hub://sappho192/drone_v2"

# overwrite=True will overwrite dataset, False will append it
vector_store = DeepLakeVectorStore(dataset_path=dataset_path, overwrite=True)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
# Create an index over the documents
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

Your Deep Lake dataset has been successfully created!


Uploading data to deeplake dataset.


100%|██████████| 88/88 [00:01<00:00, 47.85it/s]
-

Dataset(path='hub://sappho192/drone_v2', tensors=['text', 'metadata', 'embedding', 'id'])

  tensor      htype      shape      dtype  compression
  -------    -------    -------    -------  ------- 
   text       text      (88, 1)      str     None   
 metadata     json      (88, 1)      str     None   
 embedding  embedding  (88, 1536)  float32   None   
    id        text      (88, 1)      str     None   


In [28]:
import deeplake
ds = deeplake.load(dataset_path)  # Load the dataset

-

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/sappho192/drone_v2



/

hub://sappho192/drone_v2 loaded successfully.



In [29]:
import json
import pandas as pd
import numpy as np

# Assuming 'ds' is your loaded Deep Lake dataset

# Create a dictionary to hold the data
data = {}

# Iterate through the tensors in the dataset
for tensor_name in ds.tensors:
    tensor_data = ds[tensor_name].numpy()

    # Check if the tensor is multi-dimensional
    if tensor_data.ndim > 1:
        # Flatten multi-dimensional tensors
        data[tensor_name] = [np.array(e).flatten().tolist() for e in tensor_data]
    else:
        # Convert 1D tensors directly to lists and decode text
        if tensor_name == "text":
            data[tensor_name] = [t.tobytes().decode('utf-8') if t else "" for t in tensor_data]
        else:
            data[tensor_name] = tensor_data.tolist()

# Create a Pandas DataFrame from the dictionary
df = pd.DataFrame(data)

In [30]:
# Function to display a selected record
def display_record(record_number):
    record = df.iloc[record_number]
    display_data = {
        "ID": record.get("id", "N/A"),
        "Metadata": record.get("metadata", "N/A"),
        "Text": record.get("text", "N/A"),
        "Embedding": record.get("embedding", "N/A")
    }

    # Print the ID
    print("ID:")
    print(display_data["ID"])
    print()

    # Print the metadata in a structured format
    print("Metadata:")
    metadata = display_data["Metadata"]
    if isinstance(metadata, list):
        for item in metadata:
            for key, value in item.items():
                print(f"{key}: {value}")
            print()
    else:
        print(metadata)
    print()

    # Print the text
    print("Text:")
    print(display_data["Text"])
    print()

    # Print the embedding
    print("Embedding:")
    print(display_data["Embedding"])
    print()

# Function call to display a record
rec = 0  # Replace with the desired record number
display_record(rec)

ID:
['989486a2-795b-4115-bebb-d47e82b284fc']

Metadata:
file_path: /home/tikim/code/RAG-Driven-Generative-AI/Chapter03/data/1804.06985.txt
file_name: 1804.06985.txt
file_type: text/plain
file_size: 3798
creation_date: 2025-04-23
last_modified_date: 2025-04-23
_node_content: {"id_": "989486a2-795b-4115-bebb-d47e82b284fc", "embedding": null, "metadata": {"file_path": "/home/tikim/code/RAG-Driven-Generative-AI/Chapter03/data/1804.06985.txt", "file_name": "1804.06985.txt", "file_type": "text/plain", "file_size": 3798, "creation_date": "2025-04-23", "last_modified_date": "2025-04-23"}, "excluded_embed_metadata_keys": ["file_name", "file_type", "file_size", "creation_date", "last_modified_date", "last_accessed_date"], "excluded_llm_metadata_keys": ["file_name", "file_type", "file_size", "creation_date", "last_modified_date", "last_accessed_date"], "relationships": {"1": {"node_id": "2e2f6b92-57c2-4663-af82-31c529072c09", "node_type": "4", "metadata": {"file_path": "/home/tikim/code/RAG-Drive

# Original documents

In [31]:
# Ensure 'text' column is of type string
df['text'] = df['text'].astype(str)
# Create documents with IDs
documents = [Document(text=row['text'], doc_id=str(row['id'])) for _, row in df.iterrows()]

# Pipeline 3:Index-based RAG

## User input and RAG parameters

In [32]:
user_input="How do drones identify vehicles?"

#similarity_top_k
k=3
#temperature
temp=0.1
#num_output
mt=1024

## Cosine similarity metric

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_cosine_similarity_with_embeddings(text1, text2):
    embeddings1 = model.encode(text1)
    embeddings2 = model.encode(text2)
    similarity = cosine_similarity([embeddings1], [embeddings2])
    return similarity[0][0]

/home/tikim/code/RAG-Driven-Generative-AI/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/tikim/code/RAG-Driven-Generative-AI/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


# Vector store index query engine

In [35]:
from llama_index.core import VectorStoreIndex
vector_store_index = VectorStoreIndex.from_documents(documents)

In [36]:
print(type(vector_store_index))

<class 'llama_index.core.indices.vector_store.base.VectorStoreIndex'>


In [37]:
vector_query_engine = vector_store_index.as_query_engine(similarity_top_k=k, temperature=temp, num_output=mt)

In [38]:
print(type(vector_query_engine))

<class 'llama_index.core.query_engine.retriever_query_engine.RetrieverQueryEngine'>


## Query response and source

In [39]:
import pandas as pd
import textwrap

def index_query(input_query):
    response = vector_query_engine.query(input_query)

    # Optional: Print a formatted view of the response (remove if you don't need it in the output)
    print(textwrap.fill(str(response), 100))

    node_data = []
    for node_with_score in response.source_nodes:
        node = node_with_score.node
        node_info = {
            'Node ID': node.id_,
            'Score': node_with_score.score,
            'Text': node.text
        }
        node_data.append(node_info)

    df = pd.DataFrame(node_data)

    # Instead of printing, return the DataFrame and the response object
    return df, response


In [40]:
import time
#start the timer
start_time = time.time()
df, response = index_query(user_input)
# Stop the timer
end_time = time.time()
# Calculate and print the execution time
elapsed_time = end_time - start_time
print(f"Query execution time: {elapsed_time:.4f} seconds")

print(df.to_markdown(index=False, numalign="left", stralign="left"))  # Display the DataFrame using markdown

Drones can automatically track and detect vehicles using deep learning based machine learning
algorithms. Additionally, drones can automatically identify vehicles across different cameras with
different viewpoints and hardware specifications using reidentification methods.
Query execution time: 2.8989 seconds
| Node ID                              | Score    | Text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

Node information and relationships

In [41]:
nodeid=response.source_nodes[0].node_id
nodeid

'cab090a7-756f-4d83-985f-73a9d138a19a'

In [42]:
response.source_nodes[0].get_text()

"['Automatic tracking and detection of UAVs from commercial cameras have become accurate thanks to the development of deep learning based machine learning algorithms.  219  It is also possible to automatically identify UAVs across different cameras with different viewpoints and hardware specification with reidentification methods.  220  Commercial systems such as the Aaronia AARTOS have been installed on major international airports.  221   222  Once a UAV is detected it can be countered with kinetic force missiles projectiles or another UAV or by nonkinetic force laser microwaves communications jamming.  223  Antiaircraft missile systems such as the Iron Dome are also being enhanced with CUAS technologies. Utilising a smart UAV swarm to counter one or more hostile UAVs is also proposed.  224  Regulation  edit  Main article Regulation of unmanned aerial vehicles Regulatory bodies around the world are developing unmanned aircraft system traffic management solutions to better integrate U

## Optimized chunking

In [43]:
# Assuming you have the 'response' object from query_engine.query()

for node_with_score in response.source_nodes:
    node = node_with_score.node  # Extract the Node object from NodeWithScore
    chunk_size = len(node.text)
    print(f"Node ID: {node.id_}, Chunk Size: {chunk_size} characters")

Node ID: cab090a7-756f-4d83-985f-73a9d138a19a, Chunk Size: 4737 characters
Node ID: 5db51ce2-a696-4490-b7d6-e4462cd8ccd2, Chunk Size: 3384 characters
Node ID: 47626cfa-bdbe-4ab2-899f-8b9ff31b8347, Chunk Size: 4686 characters


## Performance metric

In [44]:
import numpy as np

def info_metrics(response):
  # Calculate the performance (handling None scores)
  scores = [node.score for node in response.source_nodes if node.score is not None]
  if scores:  # Check if there are any valid scores
      weights = np.exp(scores) / np.sum(np.exp(scores))
      perf = np.average(scores, weights=weights) / elapsed_time
  else:
      perf = 0  # Or some other default value if all scores are None

  average_score=np.average(scores, weights=weights)
  print(f"Average score: {average_score:.4f}")
  print(f"Query execution time: {elapsed_time:.4f} seconds")
  print(f"Performance metric: {perf:.4f}")

In [45]:
info_metrics(response)

Average score: 0.8353
Query execution time: 2.8989 seconds
Performance metric: 0.2881


# Tree index query engine

In [46]:
from llama_index.core import TreeIndex
tree_index = TreeIndex.from_documents(documents)

In [47]:
print(type(tree_index))

<class 'llama_index.core.indices.tree.base.TreeIndex'>


In [48]:
tree_query_engine = tree_index.as_query_engine(similarity_top_k=k, temperature=temp, num_output=mt)

In [49]:
import time
import textwrap
# Start the timer
start_time = time.time()
response = tree_query_engine.query(user_input)
# Stop the timer
end_time = time.time()
# Calculate and print the execution time
elapsed_time = end_time - start_time
print(f"Query execution time: {elapsed_time:.4f} seconds")

print(textwrap.fill(str(response), 100))

Query execution time: 3.6261 seconds
Drones identify vehicles by utilizing computer vision technology related to object detection. This
involves detecting instances of semantic objects of a certain class, such as cars, in digital images
and videos. The process may include using deep neural network models trained on datasets like COCO
to detect objects of common classes. Additionally, drones can employ techniques like image
annotation, vehicle counting, and tracking objects to identify vehicles in their surroundings.


## Performance metric

In [50]:
similarity_score = calculate_cosine_similarity_with_embeddings(user_input, str(response))
print(f"Cosine Similarity Score: {similarity_score:.3f}")
print(f"Query execution time: {elapsed_time:.4f} seconds")
performance=similarity_score/elapsed_time
print(f"Performance metric: {performance:.4f}")

Cosine Similarity Score: 0.743
Query execution time: 3.6261 seconds
Performance metric: 0.2048


# List index query engine

In [51]:
from llama_index.core import ListIndex
list_index = ListIndex.from_documents(documents)

In [52]:
print(type(list_index))

<class 'llama_index.core.indices.list.base.SummaryIndex'>


In [53]:
list_query_engine = list_index.as_query_engine(similarity_top_k=k, temperature=temp, num_output=mt)

In [54]:
#start the timer
start_time = time.time()
response = list_query_engine.query(user_input)
# Stop the timer
end_time = time.time()
# Calculate and print the execution time
elapsed_time = end_time - start_time
print(f"Query execution time: {elapsed_time:.4f} seconds")

print(textwrap.fill(str(response), 100))

Query execution time: 21.5347 seconds
Drones can identify vehicles through computer vision techniques, which involve processing images or
video captured by the drone's camera. The computer vision system analyzes the visual data to detect
and recognize vehicles by performing tasks like object detection, segmentation, and classification.
This process allows the system to learn and identify specific features or patterns that represent
vehicles in the images or video frames, often utilizing convolutional neural networks (CNNs) for
effective processing of visual data and extraction of relevant information for vehicle
identification.


## Performance metric

In [55]:
similarity_score = calculate_cosine_similarity_with_embeddings(user_input, str(response))
print(f"Cosine Similarity Score: {similarity_score:.3f}")
print(f"Query execution time: {elapsed_time:.4f} seconds")
performance=similarity_score/elapsed_time
print(f"Performance metric: {performance:.4f}")

Cosine Similarity Score: 0.748
Query execution time: 21.5347 seconds
Performance metric: 0.0347


# Keyword index query index

In [56]:
from llama_index.core import KeywordTableIndex
keyword_index = KeywordTableIndex.from_documents(documents)

In [57]:
# Extract data for DataFrame
data = []
for keyword, doc_ids in keyword_index.index_struct.table.items():
    for doc_id in doc_ids:
        data.append({"Keyword": keyword, "Document ID": doc_id})

# Create the DataFrame
df = pd.DataFrame(data)
df

,Keyword,Document ID
0,binary,180e6da4-0a18-47c7-aff9-c21859f835be
1,kerr,180e6da4-0a18-47c7-aff9-c21859f835be
2,entropy,180e6da4-0a18-47c7-aff9-c21859f835be
3,high energy physics,180e6da4-0a18-47c7-aff9-c21859f835be
4,theory,180e6da4-0a18-47c7-aff9-c21859f835be
...,...,...
4586,vancouver island university,c08ca894-a017-42da-bd2e-7ec6abc2a250
4587,arts,c08ca894-a017-42da-bd2e-7ec6abc2a250
4588,great,c08ca894-a017-42da-bd2e-7ec6abc2a250
4589,humanities,c08ca894-a017-42da-bd2e-7ec6abc2a250


In [58]:
keyword_query_engine = keyword_index.as_query_engine(similarity_top_k=k, temperature=temp, num_output=mt)

In [59]:
import time

# Start the timer
start_time = time.time()

# Execute the query (using .query() method)
response = keyword_query_engine.query(user_input)

# Stop the timer
end_time = time.time()

# Calculate and print the execution time
elapsed_time = end_time - start_time
print(f"Query execution time: {elapsed_time:.4f} seconds")

print(textwrap.fill(str(response), 100))

Query execution time: 0.9488 seconds
Drones can identify vehicles through various methods such as using onboard cameras and sensors to
capture images and data, which are then processed through computer vision algorithms to recognize
and classify vehicles based on their visual characteristics. Additionally, drones can utilize re-
identification methods across different cameras with different viewpoints and hardware
specifications to accurately identify vehicles.


## Performance metric

In [60]:
similarity_score = calculate_cosine_similarity_with_embeddings(user_input, str(response))
print(f"Cosine Similarity Score: {similarity_score:.3f}")
print(f"Query execution time: {elapsed_time:.4f} seconds")
performance=similarity_score/elapsed_time
print(f"Performance metric: {performance:.4f}")

Cosine Similarity Score: 0.804
Query execution time: 0.9488 seconds
Performance metric: 0.8472
